In [2]:
import requests
import pandas as pd
import time
import urllib.parse

c:\Users\ruedi.luethi\.conda\envs\streamlit\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [3]:
GENERATION_TO_REGION = {
    "generation-i": "Kanto",
    "generation-ii": "Johto",
    "generation-iii": "Hoenn",
    "generation-iv": "Sinnoh",
    "generation-v": "Unova",
    "generation-vi": "Kalos",
    "generation-vii": "Alola",
    "generation-viii": "Galar",
    "generation-ix": "Paldea",
}

TYPE_DE = {
    "normal": "Normal", "fire": "Feuer", "water": "Wasser", "electric": "Elektro",
    "grass": "Pflanze", "ice": "Eis", "fighting": "Kampf", "poison": "Gift",
    "ground": "Boden", "flying": "Flug", "psychic": "Psycho", "bug": "Käfer",
    "rock": "Gestein", "ghost": "Geist", "dragon": "Drache", "dark": "Unlicht",
    "steel": "Stahl", "fairy": "Fee",
}

In [5]:

def get_json(url, retries=3):
    for i in range(retries):
        try:
            r = requests.get(url, timeout=10)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if i == retries - 1:
                print(f"  Fehler bei {url}: {e}")
                return None
            time.sleep(1)

BULBA_API = "https://bulbapedia.bulbagarden.net/w/api.php"

def get_bulbapedia_sprite_urls(entries):
    """
    entries: list of (pokemon_id, name_en)
    Gibt dict {pokemon_id: url} zurück.
    Bulbapedia Dateiname: z.B. "0004Charmander.png"
    MediaWiki API erlaubt bis zu 50 Titles pro Request.
    """
    result = {}
    # Aufbauen: {filename: pokemon_id}
    filenames = {}
    for pid, name_en in entries:
        # Ersten Buchstaben gross, rest klein — Bulbapedia-Konvention
        name_cap = name_en.capitalize()
        filename = f"File:{pid:04d}{name_cap}.png"
        filenames[filename] = pid

    # In Batches von 50
    keys = list(filenames.keys())
    for i in range(0, len(keys), 50):
        batch = keys[i:i+50]
        titles = "|".join(batch)
        params = {
            "action": "query",
            "titles": titles,
            "prop": "imageinfo",
            "iiprop": "url",
            "format": "json",
        }
        try:
            r = requests.get(BULBA_API, params=params, timeout=15)
            data = r.json()
            pages = data.get("query", {}).get("pages", {})
            for page in pages.values():
                title = page.get("title", "")
                imageinfo = page.get("imageinfo", [])
                if imageinfo:
                    url = imageinfo[0].get("url", "")
                    pid = filenames.get(title)
                    if pid and url:
                        result[pid] = url
        except Exception as e:
            print(f"  Bulbapedia Fehler bei Batch {i}: {e}")
        time.sleep(0.2)

    return result

def get_de_name(names):
    for n in names:
        if n["language"]["name"] == "de":
            return n["name"]
    return None

def parse_evolution_chain(chain, result=None, stage=0):
    """Gibt Liste von (name, stage) zurück"""
    if result is None:
        result = []
    result.append((chain["species"]["name"], stage))
    for evo in chain.get("evolves_to", []):
        parse_evolution_chain(evo, result, stage + 1)
    return result

In [9]:
# Cache für Evolution Chains
evo_cache = {}

def get_evolution_chain(url):
    if url in evo_cache:
        return evo_cache[url]
    data = get_json(url)
    if not data:
        return []
    chain = parse_evolution_chain(data["chain"])
    evo_cache[url] = chain
    return chain

# Wie viele Pokémon holen? (1025 = alle, weniger zum Testen)
TOTAL = 1025

records = []

print(f"Hole {TOTAL} Pokémon von der PokeAPI...")

for pokemon_id in range(1, TOTAL + 1):
    if pokemon_id % 50 == 0:
        print(f"  {pokemon_id}/{TOTAL}...")

    # Call 1: Species → DE-Name, Generation, Region, Evo-Chain-URL
    species = get_json(f"https://pokeapi.co/api/v2/pokemon-species/{pokemon_id}/")
    if not species:
        continue

    name_de = get_de_name(species.get("names", []))
    name_en = next((n["name"] for n in species["names"] if n["language"]["name"] == "en"), None)
    generation = species.get("generation", {}).get("name", "")
    region = GENERATION_TO_REGION.get(generation, "")
    evo_chain_url = species.get("evolution_chain", {}).get("url", "")
    evolves_from = species.get("evolves_from_species", {})
    evolves_from_en = evolves_from["name"] if evolves_from else None

    # Entwicklungsstufe bestimmen
    evo_chain = get_evolution_chain(evo_chain_url) if evo_chain_url else []
    evo_stage = 0
    evo_names_en = [e[0] for e in evo_chain]
    for (evo_name, stage) in evo_chain:
        if evo_name == species["name"]:
            evo_stage = stage
            break

    # Call 2: Pokemon → Typen, Stats, Sprite
    pokemon = get_json(f"https://pokeapi.co/api/v2/pokemon/{pokemon_id}/")
    if not pokemon:
        continue

    types = [t["type"]["name"] for t in pokemon.get("types", [])]
    type1_de = TYPE_DE.get(types[0], types[0]) if len(types) > 0 else ""
    type2_de = TYPE_DE.get(types[1], types[1]) if len(types) > 1 else ""

    stats = {s["stat"]["name"]: s["base_stat"] for s in pokemon.get("stats", [])}

    sprite_url = (
        pokemon.get("sprites", {})
        .get("other", {})
        .get("official-artwork", {})
        .get("front_default", "")
    )

    records.append({
        "nr": pokemon_id,
        "name_de": name_de,
        "name_en": name_en,
        "typ1": type1_de,
        "typ2": type2_de,
        "generation": generation.replace("generation-", "").upper(),
        "region": region,
        "entwicklungsstufe": evo_stage,
        "entwickelt_aus_en": evolves_from_en,
        "hp": stats.get("hp", ""),
        "angriff": stats.get("attack", ""),
        "verteidigung": stats.get("defense", ""),
        "spezial_angriff": stats.get("special-attack", ""),
        "spezial_verteidigung": stats.get("special-defense", ""),
        "initiative": stats.get("speed", ""),
        "sprite_url": sprite_url,
    })

    time.sleep(0.05)  # Rate limiting

df = pd.DataFrame(records)

# Deutschen Namen von evolves_from dazujointen
name_map = df.set_index("name_en")["name_de"].to_dict()
df["entwickelt_aus_de"] = df["entwickelt_aus_en"].map(name_map)

# Bulbapedia Sprites holen (in Batches)
print("\nHole Bulbapedia Sprites...")
entries = list(zip(df["nr"], df["name_en"]))
bulba_urls = get_bulbapedia_sprite_urls(entries)
df["sprite_bulbapedia"] = df["nr"].map(bulba_urls)
found = df["sprite_bulbapedia"].notna().sum()
print(f"  {found}/{len(df)} Bulbapedia Sprites gefunden")

# Spalten ordnen
cols = [
    "nr", "name_de", "name_en", "typ1", "typ2",
    "region", "generation", "entwicklungsstufe",
    "entwickelt_aus_de", "entwickelt_aus_en",
    "hp", "angriff", "verteidigung",
    "spezial_angriff", "spezial_verteidigung", "initiative",
    "sprite_url", "sprite_bulbapedia"
]
df = df[cols]

output_path = "../data/pokemon/pokedex.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"\nFertig! {len(df)} Pokémon gespeichert → {output_path}")
print(df.head(10).to_string())


Hole 1025 Pokémon von der PokeAPI...
  50/1025...
  100/1025...
  150/1025...
  200/1025...
  250/1025...
  300/1025...
  350/1025...
  400/1025...
  450/1025...
  500/1025...
  550/1025...
  600/1025...
  650/1025...
  700/1025...
  750/1025...
  800/1025...
  850/1025...
  900/1025...
  950/1025...
  1000/1025...

Hole Bulbapedia Sprites...
  989/1025 Bulbapedia Sprites gefunden

Fertig! 1025 Pokémon gespeichert → ../data/pokemon/pokedex.csv
   nr    name_de     name_en     typ1  typ2 region generation  entwicklungsstufe entwickelt_aus_de entwickelt_aus_en  hp  angriff  verteidigung  spezial_angriff  spezial_verteidigung  initiative                                                                                              sprite_url                                                      sprite_bulbapedia
0   1    Bisasam   Bulbasaur  Pflanze  Gift  Kanto          I                  0               NaN              None  45       49            49               65                    65

In [21]:
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd
import requests

# Input/Output paths
csv_path = Path("../data/pokemon/pokedex.csv")
out_dir = Path("../data/pokemon/sprites")
out_dir.mkdir(parents=True, exist_ok=True)

# Load Pokedex CSV
if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found: {csv_path.resolve()}")

df = pd.read_csv(csv_path, encoding="utf-8-sig")
if "sprite_bulbapedia" not in df.columns:
    raise KeyError("Column 'sprite_bulbapedia' not found in pokedex.csv")

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (compatible; sprite-downloader/1.0)"})

downloaded = 0
skipped = 0
failed = 0

for _, row in df.iterrows():
    url = row.get("sprite_bulbapedia")
    if pd.isna(url) or not str(url).strip():
        skipped += 1
        continue

    url = str(url).strip()

    # Build a deterministic filename: 4-digit pokedex number + original extension
    pokemon_id = int(row["nr"]) if "nr" in row and not pd.isna(row["nr"]) else None
    parsed = urlparse(url)
    ext = Path(parsed.path).suffix.lower() or ".png"

    if pokemon_id is not None:
        filename = f"{pokemon_id:04d}{ext}"
    else:
        fallback_name = Path(parsed.path).name or f"sprite_{downloaded + failed + skipped:04d}{ext}"
        filename = fallback_name

    target = out_dir / filename

    # Skip already downloaded files
    if target.exists() and target.stat().st_size > 0:
        skipped += 1
        continue

    print(f"Downloading {url} → {target.name}...")

    try:
        resp = session.get(url, timeout=20)
        resp.raise_for_status()

        content_type = resp.headers.get("Content-Type", "")
        if not content_type.startswith("image/"):
            raise ValueError(f"URL did not return an image: {content_type}")

        target.write_bytes(resp.content)
        downloaded += 1
    except Exception as e:
        failed += 1
        print(f"Fehler bei {url}: {e}")

print(f"Fertig. Downloaded={downloaded}, Skipped={skipped}, Failed={failed}")
print(f"Sprites gespeichert in: {out_dir.resolve()}")

Fertig. Downloaded=989, Skipped=36, Failed=0
Sprites gespeichert in: C:\work\math-playground\data\pokemon\sprites


In [2]:
from pathlib import Path
import pandas as pd

csv_path = Path("../data/pokemon/pokedex.csv")
out_dir = Path("../data/pokemon/sprites")
out_dir.mkdir(parents=True, exist_ok=True)

# Load Pokedex CSV
if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found: {csv_path.resolve()}")

df = pd.read_csv(csv_path, encoding="utf-8-sig")

# --- Recursive evolution lookup from entwickelt_aus_en ---
# Normalize keys to avoid case/whitespace mismatches.
name_norm = df["name_en"].astype(str).str.strip().str.lower()
parent_norm = df["entwickelt_aus_en"].fillna("").astype(str).str.strip().str.lower()

df_lookup = pd.DataFrame({"name": name_norm, "parent": parent_norm})

# child -> parent mapping from CSV
parent_of = {
    row["name"]: row["parent"]
    for _, row in df_lookup.iterrows()
    if row["name"]
}

# Build parent -> [children] mapping
children_of = {}
for child, parent in parent_of.items():
    if not parent:
        continue
    children_of.setdefault(parent, []).append(child)

for parent in children_of:
    children_of[parent].sort()


def collect_leaf_paths(name, path=None):
    if path is None:
        path = []
    path = path + [name]
    children = children_of.get(name, [])
    if not children:
        return [path]

    paths = []
    for child in children:
        paths.extend(collect_leaf_paths(child, path))
    return paths


def full_evolution_sequence(name):
    if pd.isna(name) or not str(name).strip():
        return []

    name = str(name).strip().lower()

    # Move to root (first stage) by following parents upward
    root = name
    seen_up = set()
    while True:
        if root in seen_up:
            break
        seen_up.add(root)

        parent = parent_of.get(root, "")
        if not parent:
            break
        root = parent

    # Build all root->leaf paths recursively
    paths = collect_leaf_paths(root)

    # Keep only paths that include current pokemon
    relevant_paths = [p for p in paths if name in p]
    if not relevant_paths:
        return []

    # Only keep chains with at least 2 stages.
    if not any(len(p) > 1 for p in relevant_paths):
        return []

    # Return as list object in df (merged unique order for branch evolutions)
    merged = []
    for path in relevant_paths:
        for p in path:
            if p not in merged:
                merged.append(p)
    return merged


df["entwicklungsabfolge_en"] = df["name_en"].apply(full_evolution_sequence)

# Optional DE list version
de_name_of = dict(zip(df["name_en"].astype(str).str.strip().str.lower(), df["name_de"]))


def to_de_list(seq_en):
    return [de_name_of.get(n, n) for n in seq_en] if isinstance(seq_en, list) else []


df["entwicklungsabfolge_de"] = df["entwicklungsabfolge_en"].apply(to_de_list)

# Additional list with pokedex numbers for the evolution sequence
nr_of = dict(zip(df["name_en"].astype(str).str.strip().str.lower(), df["nr"]))


def to_nr_list(seq_en):
    if not isinstance(seq_en, list):
        return []
    return [int(nr_of[n]) for n in seq_en if n in nr_of and pd.notna(nr_of[n])]


df["entwicklungsabfolge_nr"] = df["entwicklungsabfolge_en"].apply(to_nr_list)

# Show first rows for quick validation
# print(df[["nr", "name_en", "entwickelt_aus_en", "entwicklungsabfolge_en", "entwicklungsabfolge_nr"]].head(10).to_string(index=False))
df.head()


# df['typ1'].value_counts()
# df['typ2'].value_counts()

,nr,name_de,name_en,typ1,typ2,region,generation,entwicklungsstufe,entwickelt_aus_de,entwickelt_aus_en,...,angriff,verteidigung,spezial_angriff,spezial_verteidigung,initiative,sprite_url,sprite_bulbapedia,entwicklungsabfolge_en,entwicklungsabfolge_de,entwicklungsabfolge_nr
0,1,Bisasam,Bulbasaur,Pflanze,Gift,Kanto,I,0,NaN,NaN,...,49,49,65,65,45,https://raw.githubusercontent.com/PokeAPI/spri...,https://archives.bulbagarden.net/media/upload/...,"[bulbasaur, ivysaur, venusaur]","[Bisasam, Bisaknosp, Bisaflor]","[1, 2, 3]"
1,2,Bisaknosp,Ivysaur,Pflanze,Gift,Kanto,I,1,NaN,bulbasaur,...,62,63,80,80,60,https://raw.githubusercontent.com/PokeAPI/spri...,https://archives.bulbagarden.net/media/upload/...,"[bulbasaur, ivysaur, venusaur]","[Bisasam, Bisaknosp, Bisaflor]","[1, 2, 3]"
2,3,Bisaflor,Venusaur,Pflanze,Gift,Kanto,I,2,NaN,ivysaur,...,82,83,100,100,80,https://raw.githubusercontent.com/PokeAPI/spri...,https://archives.bulbagarden.net/media/upload/...,"[bulbasaur, ivysaur, venusaur]","[Bisasam, Bisaknosp, Bisaflor]","[1, 2, 3]"
3,4,Glumanda,Charmander,Feuer,NaN,Kanto,I,0,NaN,NaN,...,52,43,60,50,65,https://raw.githubusercontent.com/PokeAPI/spri...,https://archives.bulbagarden.net/media/upload/...,"[charmander, charmeleon, charizard]","[Glumanda, Glutexo, Glurak]","[4, 5, 6]"
4,5,Glutexo,Charmeleon,Feuer,NaN,Kanto,I,1,NaN,charmander,...,64,58,80,65,80,https://raw.githubusercontent.com/PokeAPI/spri...,https://archives.bulbagarden.net/media/upload/...,"[charmander, charmeleon, charizard]","[Glumanda, Glutexo, Glurak]","[4, 5, 6]"


In [7]:
# Neues DataFrame mit den gewuenschten Spalten
df_new = df[[
    "nr",
    "name_de",
    "name_en",
    "typ1",
    "typ2",
    "region",
    "generation",
    "entwicklungsstufe",
    "entwickelt_aus_en",
    "sprite_url",
    "sprite_bulbapedia",
    "entwicklungsabfolge_nr",
]].copy()

# Fuer JSON-Export: zusaetzliche Verschachtelung [] um jede Liste
df_new_export = df_new.copy()
df_new_export["entwicklungsabfolge_nr"] = df_new_export["entwicklungsabfolge_nr"].apply(
    lambda seq: [seq] if isinstance(seq, list) else [[]]
)

json_path = "../data/pokemon/pokedex_new.json"
df_new_export.to_json(json_path, orient="records", force_ascii=False, indent=2)

In [20]:
# json_path = "../data/pokemon/pokedex_edited.json"
json_path = "../data/pokemon/pokedex_extended_cleaned.json"
df_edited = pd.read_json(json_path, orient="records")

out_dir = Path("../data/pokemon/sprites")
# df_edited.iloc[15:25].head()

for k_, pokemon in df_edited.iterrows():
    forms = pokemon.get("weitere_erscheinungsformen")
    # if pd.notna(pokemon["weitere_erscheinungsformen"]):
    if isinstance(forms, list) and len(forms) > 0:
        print(f"{pokemon['name_de']} hat weitere Erscheinungsformen:")
        pokemon_id = int(pokemon["nr"])

        for form in pokemon["weitere_erscheinungsformen"]:
            sprite_path = out_dir / f"{pokemon_id:04d}_{form['form_name']}.png"
            if not sprite_path.exists():
                r = requests.get(form["sprite_bulbapedia"], timeout=20)
                r.raise_for_status()
                sprite_path.write_bytes(r.content)
                print(f"geladen: {sprite_path}")
            else:
                print(f"existiert: {sprite_path}")

Bisaflor hat weitere Erscheinungsformen:
geladen: ..\data\pokemon\sprites\0003_Mega.png
geladen: ..\data\pokemon\sprites\0003_Gigantamax.png
Glurak hat weitere Erscheinungsformen:
geladen: ..\data\pokemon\sprites\0006_Mega_X.png
geladen: ..\data\pokemon\sprites\0006_Mega_Y.png
geladen: ..\data\pokemon\sprites\0006_Gigantamax.png
Turtok hat weitere Erscheinungsformen:
geladen: ..\data\pokemon\sprites\0009_Mega.png
geladen: ..\data\pokemon\sprites\0009_Gigantamax.png
Smettbo hat weitere Erscheinungsformen:
geladen: ..\data\pokemon\sprites\0012_Gigantamax.png
Bibor hat weitere Erscheinungsformen:
geladen: ..\data\pokemon\sprites\0015_Mega.png
Tauboss hat weitere Erscheinungsformen:
geladen: ..\data\pokemon\sprites\0018_Mega.png
Rattfratz hat weitere Erscheinungsformen:
geladen: ..\data\pokemon\sprites\0019_Alola.png
Rattikarl hat weitere Erscheinungsformen:
geladen: ..\data\pokemon\sprites\0020_Alola.png
Pikachu hat weitere Erscheinungsformen:
geladen: ..\data\pokemon\sprites\0025_Cosplay

In [3]:
from fpdf import FPDF
from pathlib import Path

import math

out_dir = Path("../data/pokemon/sprites")

# json_path = "../data/pokemon/pokedex_edited.json"
json_path = "../data/pokemon/pokedex_extended_cleaned.json"
df_edited = pd.read_json(json_path, orient="records")

df_edited.iloc[15:25].head()

for k_, pokemon in df_edited.iterrows():
    more_forms = pokemon.get("weitere_erscheinungsformen")
    more_devs = pokemon.get("weitere_entwicklungen")
    all_add_forms = [more_forms, more_devs]
    for forms in all_add_forms:
        if isinstance(forms, list) and len(forms) > 0:
            pokemon_id = int(pokemon["nr"])

            for form in forms:
                sprite_path = out_dir / f"{pokemon_id:04d}_{form['form_name']}.png"
                if not sprite_path.exists():
                    r = requests.get(form["sprite_bulbapedia"], timeout=20)
                    r.raise_for_status()
                    sprite_path.write_bytes(r.content)
                    print(f"geladen: {sprite_path}")


def draw_arrow(pdf, x1, y1, x2, y2, color=(0, 0, 0), line_w=0.2, head_len=1.0, head_angle_deg=45):
    # shaft
    pdf.set_draw_color(*color)
    pdf.set_line_width(line_w)
    pdf.line(x1, y1, x2, y2)

    # arrow head (2 short lines)
    ang = math.atan2(y2 - y1, x2 - x1)
    a = math.radians(head_angle_deg)

    x3 = x2 - head_len * math.cos(ang - a)
    y3 = y2 - head_len * math.sin(ang - a)
    x4 = x2 - head_len * math.cos(ang + a)
    y4 = y2 - head_len * math.sin(ang + a)

    pdf.line(x2, y2, x3, y3)
    pdf.line(x2, y2, x4, y4)


H_MARGIN = 20
V_MARGIN = 20

TYPE_ICON_FILE = {
    "Wasser": "Pokemon_Type_Icon_Water@4x.png",
    "Normal": "Pokemon_Type_Icon_Normal@4x.png",
    "Pflanze": "Pokemon_Type_Icon_Grass@4x.png",
    "Käfer": "Pokemon_Type_Icon_Bug@4x.png",
    "Feuer": "Pokemon_Type_Icon_Fire@4x.png",
    "Psycho": "Pokemon_Type_Icon_Psychic@4x.png",
    "Elektro": "Pokemon_Type_Icon_Electric@4x.png",
    "Gestein": "Pokemon_Type_Icon_Rock@4x.png",
    "Unlicht": "Pokemon_Type_Icon_Dark@4x.png",
    "Gift": "Pokemon_Type_Icon_Poison@4x.png",
    "Boden": "Pokemon_Type_Icon_Ground@4x.png",
    "Kampf": "Pokemon_Type_Icon_Fighting@4x.png",
    "Drache": "Pokemon_Type_Icon_Dragon@4x.png",
    "Stahl": "Pokemon_Type_Icon_Steel@4x.png",
    "Geist": "Pokemon_Type_Icon_Ghost@4x.png",
    "Eis": "Pokemon_Type_Icon_Ice@4x.png",
    "Fee": "Pokemon_Type_Icon_Fairy@4x.png",
    "Flug": "Pokemon_Type_Icon_Flying@4x.png",
}


pdf = FPDF(format='A4')
pdf.set_margins(left=H_MARGIN, top=40, right=H_MARGIN)
pdf.add_font(family='Pokemon',fname=f'../data/pokemon/pokemon-ds-font.ttf',uni=True)
pdf.add_font(family='Lato',fname=f'../data/pokemon/Lato-Regular.ttf',uni=True)
pdf.add_font(family='Lato',fname=f'../data/pokemon/Lato-Bold.ttf',uni=True, style="B")
pdf.set_auto_page_break(auto=False)


def draw_pokemon_card(pdf, x, y, width, height, pokemon, sprite_path, alternative_form=False):
#     pdf.set_font("Arial", "B", 16)
#     pdf.cell(0, 10, f"{pokemon['nr']:03d} - {pokemon['name_de']}", ln=True)

    k_shift = 1
    CARD_PADDING = 10
    card_width = width - 2 * CARD_PADDING

    TEXT_SHIFT = 30

    if sprite_path.exists():
        pdf.image(str(sprite_path), x=x + CARD_PADDING, y=y + CARD_PADDING, w=card_width)

    icon_size = card_width / 7

    types = []
    if pokemon['typ1']:
        if pokemon['typ1'] in TYPE_ICON_FILE:
            types.append(pokemon['typ1'])
    if pokemon['typ2']:
        if pokemon['typ2'] in TYPE_ICON_FILE:
            types.append(pokemon['typ2'])

    icon_dir = "../data/pokemon/icons"
    for t in types:
        icon_x = x + width/2 - (len(types) * icon_size)/2 + types.index(t)*icon_size
        icon_y = y + height - TEXT_SHIFT - 1 - icon_size
        icon_file = TYPE_ICON_FILE.get(t)

        if icon_file is not None:
            pdf.image(f"{icon_dir}/{icon_file}", x=icon_x, y=icon_y, w=icon_size*0.9)

    if alternative_form:
        pdf.set_xy(x + CARD_PADDING, y + height - TEXT_SHIFT -1)
        pdf.set_font("Lato", "", 14)
        pdf.cell(w=card_width, h=10, txt=f"{pokemon['form_name']}", ln=True, align="C")

    pdf.set_xy(x + CARD_PADDING, y + height - TEXT_SHIFT + 6)
    pdf.set_font("Lato", "B", 14)
    pdf.cell(w=card_width, h=10, txt=f"{pokemon['name_de']}", ln=True, align="C")

    if not alternative_form:
        pdf.set_xy(x + CARD_PADDING, y + height - TEXT_SHIFT -1)
        pdf.set_font("Lato", "", 21)
        pdf.cell(w=card_width, h=10, txt=f"{pokemon['nr']:04d}", ln=True, align="C")

        
    
        # print(pokemon['entwicklungsabfolge_nr'])

        

        # print(pokemon['entwicklungsabfolge_nr'])

        next_evo_card_count = 6


        x_offset = 0
        y_offset = 0
        for j, evo_seq in enumerate(pokemon['entwicklungsabfolge_nr']):
            # y_offset = j * (icon_size + 5)

            if j >= 2:
                x_offset = (math.floor((j-2) / next_evo_card_count) + 1) * width
                if (j-2) % next_evo_card_count == 0:
                    k_shift += 1
                    y_offset = -(icon_size + 5) * (next_evo_card_count-2)
                    if x+x_offset > pdf.w - width - H_MARGIN:
                        y_offset += height

                if x+x_offset > pdf.w - width - H_MARGIN:
                    x_offset = -2*(width)

            for i, evo_nr in enumerate(evo_seq):
                # if i >= 3:
                #     break
                # evo_name = df_edited[df_edited["nr"] == evo_nr]["name_de"].values[0]
                # pdf.set_xy(x + CARD_PADDING, y + height - 12 + i*4)
                # pdf.set_font("Lato", "", 10)
                # pdf.cell(w=card_width, h=4, txt=f"{evo_nr:03d} - {evo_name}", ln=True, align="C")
                evo_path = out_dir / f"{evo_nr:04d}.png"
                
                evo_width = card_width / 3
                evo_x = x + x_offset + CARD_PADDING + card_width/2 - ((len(evo_seq)-0.5) * evo_width)/2 + i*evo_width
                if evo_path.exists():
                    pdf.image(str(evo_path), 
                            x=evo_x, 
                            y=y + height - TEXT_SHIFT + y_offset + 15, w=evo_width*0.5)
                
                pdf.set_xy(evo_x, y + height - TEXT_SHIFT + y_offset + 18 + evo_width*0.25)
                pdf.set_font("Lato", "", 8)
                if evo_nr < 9000:
                    pdf.cell(w=evo_width*0.6, h=4, txt=f"{evo_nr:04d}", ln=True, align="C")

                if i > 0:
                    draw_arrow(pdf, 
                            evo_x - evo_width*0.3, y + height - TEXT_SHIFT + y_offset + 15 + evo_width*0.3, 
                            evo_x - evo_width*0.1, y + height - TEXT_SHIFT + y_offset + 15 + evo_width*0.3)
            
            y_offset += icon_size + 5

    return k_shift


COLS = 3
ROWS = 3

# for k, pokemon in df.head(151).iterrows():
k = 0
# for k_, pokemon in df_edited.iloc[129:140].iterrows():

# for k_, pokemon in df_edited.iloc[0:151].iterrows(): # Kanto
# for k_, pokemon in df_edited.iloc[151:251].iterrows(): # Johto
# for k_, pokemon in df_edited.iloc[251:386].iterrows(): # Hoenn
# for k_, pokemon in df_edited.iloc[386:493].iterrows(): # Sinnoh
# for k_, pokemon in df_edited.iloc[493:649].iterrows(): # Einall
# for k_, pokemon in df_edited.iloc[649:721].iterrows(): # Kalos
# for k_, pokemon in df_edited.iloc[721:807].iterrows(): # Alola
# for k_, pokemon in df_edited.iloc[807:809].iterrows(): # Unknown
for k_, pokemon in df_edited.iloc[809:898].iterrows(): # Galar
# for k_, pokemon in df_edited.iloc[898:905].iterrows(): # Hisui
# for k_, pokemon in df_edited.iloc[905:1025].iterrows(): # Paldea

    pokemon_id = int(pokemon["nr"])
    sprite_path = out_dir / f"{pokemon_id:04d}.png"

    if k%(COLS*ROWS) == 0:
        pdf.add_page()

    i = (k % (COLS * ROWS)) % COLS
    j = (k % (COLS * ROWS)) // COLS

    print(k, i, j, sprite_path)

    x = H_MARGIN + i * (pdf.w - 2 * H_MARGIN) / COLS
    y = V_MARGIN + j * (pdf.h - 2 * V_MARGIN) / ROWS

    card_width = (pdf.w - 2 * H_MARGIN) / COLS
    card_height = (pdf.h - 2 * V_MARGIN) / ROWS

    k = k + draw_pokemon_card(pdf, x, y, width=card_width, height=card_height, pokemon=pokemon, sprite_path=sprite_path)
    
    
    # forms = pokemon.get("weitere_erscheinungsformen")

    more_forms = pokemon.get("weitere_erscheinungsformen")
    more_devs = pokemon.get("weitere_entwicklungen")
    all_add_forms = [more_forms, more_devs]
    for forms in all_add_forms:

        if isinstance(forms, list) and len(forms) > 0:
            for form in forms:

                form_sprite_path = out_dir / f"{int(pokemon['nr']):04d}_{form['form_name']}.png"

                if k%(COLS*ROWS) == 0:
                    pdf.add_page()

                i = (k % (COLS * ROWS)) % COLS
                j = (k % (COLS * ROWS)) // COLS

                print(k, i, j, form_sprite_path)

                x = H_MARGIN + i * (pdf.w - 2 * H_MARGIN) / COLS
                y = V_MARGIN + j * (pdf.h - 2 * V_MARGIN) / ROWS

                pokemon_form = pokemon.copy()
                pokemon_form["form_name"] = form['form_name'].capitalize()
                if "typ1" in form and pd.notna(form["typ1"]):
                    pokemon_form["typ1"] = form["typ1"]
                if "typ2" in form and pd.notna(form["typ2"]):
                    pokemon_form["typ2"] = form["typ2"]

                k = k + draw_pokemon_card(pdf, x, y, width=card_width, height=card_height, pokemon=pokemon_form, sprite_path=form_sprite_path, alternative_form=True)

full_path = "../data/pokemon/pokedex_edited.pdf"
pdf.output(full_path, 'F')


C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:77: DeprecationWarning: "uni" parameter is deprecated since v2.5.1, unused and will soon be removed
  pdf.add_font(family='Pokemon',fname=f'../data/pokemon/pokemon-ds-font.ttf',uni=True)
C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:78: DeprecationWarning: "uni" parameter is deprecated since v2.5.1, unused and will soon be removed
  pdf.add_font(family='Lato',fname=f'../data/pokemon/Lato-Regular.ttf',uni=True)
C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:79: DeprecationWarning: "uni" parameter is deprecated since v2.5.1, unused and will soon be removed
  pdf.add_font(family='Lato',fname=f'../data/pokemon/Lato-Bold.ttf',uni=True, style="B")


0 0 0 ..\data\pokemon\sprites\0810.png


C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:122: DeprecationWarning: The parameter "txt" has been renamed to "text" in 2.7.6
  pdf.cell(w=card_width, h=10, txt=f"{pokemon['name_de']}", ln=True, align="C")
C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:122: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(w=card_width, h=10, txt=f"{pokemon['name_de']}", ln=True, align="C")
C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:127: DeprecationWarning: The parameter "txt" has been renamed to "text" in 2.7.6
  pdf.cell(w=card_width, h=10, txt=f"{pokemon['nr']:04d}", ln=True, align="C")
C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:127: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(w=card_width, h=10, txt=f"{pokemon['nr']

1 1 0 ..\data\pokemon\sprites\0811.png
2 2 0 ..\data\pokemon\sprites\0812.png
3 0 1 ..\data\pokemon\sprites\0812_Gigantamax.png


C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:118: DeprecationWarning: The parameter "txt" has been renamed to "text" in 2.7.6
  pdf.cell(w=card_width, h=10, txt=f"{pokemon['form_name']}", ln=True, align="C")
C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:118: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(w=card_width, h=10, txt=f"{pokemon['form_name']}", ln=True, align="C")
C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:122: DeprecationWarning: The parameter "txt" has been renamed to "text" in 2.7.6
  pdf.cell(w=card_width, h=10, txt=f"{pokemon['name_de']}", ln=True, align="C")
C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:122: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(w=card_width, h=10, txt=f"{pokemon[

...

C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:127: DeprecationWarning: The parameter "txt" has been renamed to "text" in 2.7.6
  pdf.cell(w=card_width, h=10, txt=f"{pokemon['nr']:04d}", ln=True, align="C")
C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:127: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(w=card_width, h=10, txt=f"{pokemon['nr']:04d}", ln=True, align="C")


111 0 1 ..\data\pokemon\sprites\0895.png
112 1 1 ..\data\pokemon\sprites\0896.png
113 2 1 ..\data\pokemon\sprites\0897.png
114 0 2 ..\data\pokemon\sprites\0898.png
115 1 2 ..\data\pokemon\sprites\0898_Ice_Rider.png


C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:118: DeprecationWarning: The parameter "txt" has been renamed to "text" in 2.7.6
  pdf.cell(w=card_width, h=10, txt=f"{pokemon['form_name']}", ln=True, align="C")
C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:118: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(w=card_width, h=10, txt=f"{pokemon['form_name']}", ln=True, align="C")
C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:122: DeprecationWarning: The parameter "txt" has been renamed to "text" in 2.7.6
  pdf.cell(w=card_width, h=10, txt=f"{pokemon['name_de']}", ln=True, align="C")
C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:122: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(w=card_width, h=10, txt=f"{pokemon[

116 2 2 ..\data\pokemon\sprites\0898_Shadow_Rider.png


C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:118: DeprecationWarning: The parameter "txt" has been renamed to "text" in 2.7.6
  pdf.cell(w=card_width, h=10, txt=f"{pokemon['form_name']}", ln=True, align="C")
C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:118: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(w=card_width, h=10, txt=f"{pokemon['form_name']}", ln=True, align="C")
C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:122: DeprecationWarning: The parameter "txt" has been renamed to "text" in 2.7.6
  pdf.cell(w=card_width, h=10, txt=f"{pokemon['name_de']}", ln=True, align="C")
C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_21448\1731450811.py:122: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(w=card_width, h=10, txt=f"{pokemon[

In [22]:
from fpdf import FPDF
from pathlib import Path
import pandas as pd

json_path = Path("../data/pokemon/pokedex_edited.json")
if not json_path.exists():
    raise FileNotFoundError(f"Datei nicht gefunden: {json_path.resolve()}")

df_edited = pd.read_json(json_path, orient="records")
df_edited_sorted = df_edited.sort_values("name_de", key=lambda s: s.str.lower()).reset_index(drop=True)

print(df_edited_sorted.head())

H_MARGIN = 20

pdf = FPDF(format="A4")
pdf.set_margins(left=H_MARGIN, top=20, right=H_MARGIN)
pdf.add_font(family="Pokemon", fname="../data/pokemon/pokemon-ds-font.ttf", uni=True)
pdf.add_font(family="Lato", fname="../data/pokemon/Lato-Regular.ttf", uni=True)
pdf.add_font(family="Lato", fname="../data/pokemon/Lato-Bold.ttf", uni=True, style="B")
pdf.set_auto_page_break(auto=False)
pdf.add_page()

# 3-column layout settings
COLS = 3
COL_GAP = 4.0
TOP_Y = 20.0
BOTTOM_Y = pdf.h - 20.0
line_h = 9.0
header_h = 12.0

usable_w = pdf.w - 2 * H_MARGIN
col_w = (usable_w - (COLS - 1) * COL_GAP) / COLS

col_idx = 0
y = TOP_Y


def col_x(idx):
    return H_MARGIN + idx * (col_w + COL_GAP)


def next_column_or_page():
    global col_idx, y
    col_idx += 1
    if col_idx >= COLS:
        pdf.add_page()
        col_idx = 0
    y = TOP_Y


def ensure_space(required_height):
    global y
    if y + required_height > BOTTOM_Y:
        next_column_or_page()


def write_text_line(text, bold=False, size=10):
    global y
    pdf.set_xy(col_x(col_idx), y)
    pdf.set_font("Lato", "B" if bold else "", size)
    pdf.cell(w=col_w, h=line_h, txt=text, ln=0)
    y += line_h

def write_text_line_with_sprite(text, sprite_path, bold=False, size=10, line_height=line_h):
    global y
    pdf.set_xy(col_x(col_idx), y)

    if sprite_path is not None and sprite_path.exists():
        pdf.image(str(sprite_path), w=8)

    
    pdf.set_xy(col_x(col_idx)+8, y)

    pdf.set_font("Lato", "B" if bold else "", size)
    pdf.cell(w=col_w, h=line_height, txt=text, ln=0)
    y += line_height


current_letter = None
for _, row in df_edited_sorted.iterrows():
    name = str(row["name_de"]) if pd.notna(row["name_de"]) else ""
    region = str(row["region"]) if pd.notna(row["region"]) else ""
    first_letter = name[:1].upper() if name else "#"

    if first_letter != current_letter:
        current_letter = first_letter
        
        ensure_space(header_h + line_h)
        write_text_line(f"", bold=True, size=21)

        ensure_space(header_h + line_h)
        write_text_line(f"{current_letter}", bold=True, size=21)

    ensure_space(line_h)
    
    # write_text_line(f"{int(row['nr']):04d} {name}", bold=False, size=14)
    sprite_path = Path(f"../data/pokemon/sprites/{int(row['nr']):04d}.png")
    write_text_line_with_sprite(f"{int(row['nr']):04d}  {name}", sprite_path, bold=False, size=11)
    y -= 1
    # write_text_line_with_sprite(f"            {region}", None, bold=False, size=9, line_height=0)
    write_text_line_with_sprite(f"{region}", None, bold=False, size=7, line_height=0)


full_path = "../data/pokemon/pokedex_abc_index.pdf"
pdf.output(full_path, "F")
print(f"PDF gespeichert: {full_path}")

    nr   name_de     name_en     typ1     typ2  region generation  \
0  367  Aalabyss     Huntail   Wasser     None   Hoenn        III   
1   63      Abra        Abra   Psycho     None   Kanto          I   
2  359     Absol       Absol  Unlicht     None   Hoenn        III   
3  962    Adebom  Bombirdier     Flug  Unlicht  Paldea         IX   
4  503   Admurai    Samurott   Wasser     None   Unova          V   

   entwicklungsstufe entwickelt_aus_en  \
0                  1          clamperl   
1                  0              None   
2                  0              None   
3                  0              None   
4                  2            dewott   

                                          sprite_url  \
0  https://raw.githubusercontent.com/PokeAPI/spri...   
1  https://raw.githubusercontent.com/PokeAPI/spri...   
2  https://raw.githubusercontent.com/PokeAPI/spri...   
3  https://raw.githubusercontent.com/PokeAPI/spri...   
4  https://raw.githubusercontent.com/PokeAPI/spri...

C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_16996\2889806526.py:18: DeprecationWarning: "uni" parameter is deprecated since v2.5.1, unused and will soon be removed
  pdf.add_font(family="Pokemon", fname="../data/pokemon/pokemon-ds-font.ttf", uni=True)
C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_16996\2889806526.py:19: DeprecationWarning: "uni" parameter is deprecated since v2.5.1, unused and will soon be removed
  pdf.add_font(family="Lato", fname="../data/pokemon/Lato-Regular.ttf", uni=True)
C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_16996\2889806526.py:20: DeprecationWarning: "uni" parameter is deprecated since v2.5.1, unused and will soon be removed
  pdf.add_font(family="Lato", fname="../data/pokemon/Lato-Bold.ttf", uni=True, style="B")
C:\Users\ruedi.luethi\AppData\Local\Temp\ipykernel_16996\2889806526.py:62: DeprecationWarning: The parameter "txt" has been renamed to "text" in 2.7.6
  pdf.cell(w=col_w, h=line_h, txt=text, ln=0)
C:\Users\ruedi.luethi\AppData\

PDF gespeichert: ../data/pokemon/pokedex_abc_index.pdf


In [1]:
from pathlib import Path
from urllib.parse import urljoin, unquote
import re
import requests
import pandas as pd

BASE_URL = "https://bulbapedia.bulbagarden.net"
html_path = Path("../data/pokemon/bulpedia_list.html")

if not html_path.exists():
    raise FileNotFoundError(f"Datei nicht gefunden: {html_path.resolve()}")

html = html_path.read_text(encoding="utf-8", errors="ignore")

# Alle href="/wiki/..." oder href='/wiki/...' extrahieren
pattern = r"href\s*=\s*[\"'](/wiki/[^\"'#?]+)[\"']"
matches = re.findall(pattern, html)

# Deduplizieren, Reihenfolge beibehalten
wiki_links = list(dict.fromkeys(matches))
print(f"Gefundene /wiki/-Links: {len(wiki_links)}")

def extract_table_block(page_html: str, start_pos: int) -> str:
    """Extrahiert den kompletten <table>...</table>-Block inklusive verschachtelter Tabellen."""
    tag_re = re.compile(r"</?table\b[^>]*>", re.IGNORECASE)
    depth = 0

    for m in tag_re.finditer(page_html, pos=start_pos):
        tag = m.group(0).lower()
        if tag.startswith("</table"):
            depth -= 1
            if depth == 0:
                return page_html[start_pos:m.end()]
        else:
            depth += 1

    return ""

def find_roundy_infobox(page_html: str) -> str:
    """Findet die erste Tabelle mit den Klassen 'roundy' und 'infobox'."""
    table_start_re = re.compile(
        r"<table\b[^>]*class\s*=\s*([\"'])([^\"']*)\1[^>]*>",
        re.IGNORECASE,
    )

    for m in table_start_re.finditer(page_html):
        classes = m.group(2).lower()
        if "roundy" in classes and "infobox" in classes:
            return extract_table_block(page_html, m.start())

    return ""

def extract_file_links_from_infobox(infobox_html: str) -> list[str]:
    """Extrahiert /wiki/File:... Links aus der Infobox in Original-Reihenfolge, ohne Duplikate."""
    file_link_re = re.compile(
        r"<a\b[^>]*href\s*=\s*[\"'](/wiki/File:[^\"'#?]+)[\"'][^>]*>",
        re.IGNORECASE,
    )
    links = file_link_re.findall(infobox_html)
    return list(dict.fromkeys(links))

def file_stem_from_wiki_file_link(link: str) -> str:
    """/wiki/File:0065Alakazam-Mega.png -> 0065Alakazam-Mega"""
    if not link.startswith("/wiki/File:"):
        return ""
    filename = unquote(link[len("/wiki/File:"):])
    return filename.rsplit(".", 1)[0]

def select_relevant_other_thumbnails(main_thumbnail: str, other_thumbnails: list[str]) -> list[str]:
    """
    Liefert alle relevanten Zusatzformen (nicht nur die erste).
    Heuristik: Zusatzbild muss den gleichen Basisstamm wie das Hauptbild haben
    und danach einen Trenner '-' oder '_' tragen (z.B. -Mega, -Gmax, -Alola).
    """
    if not main_thumbnail or not other_thumbnails:
        return []

    main_stem = file_stem_from_wiki_file_link(main_thumbnail)
    if not main_stem:
        return []

    # Basisstamm: vor erster Form-Erweiterung abschneiden
    # z.B. 0025Pikachu-Gmax -> 0025Pikachu
    main_base = re.split(r"[-_]", main_stem, maxsplit=1)[0]

    relevant = []
    for link in other_thumbnails:
        stem = file_stem_from_wiki_file_link(link)
        if not stem:
            continue

        if stem.startswith(main_base + "-") or stem.startswith(main_base + "_"):
            relevant.append(link)

    return relevant

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (compatible; pokedex-form-scan/1.0)"})

results = []
errors = []

# Optional: zum schnellen Testen limitieren (z.B. 100). None = alle Links
SAMPLE_LIMIT = None
links_to_process = wiki_links if SAMPLE_LIMIT is None else wiki_links[:SAMPLE_LIMIT]

for idx, wiki_path in enumerate(links_to_process, start=1):
    page_url = urljoin(BASE_URL, wiki_path)

    try:
        resp = session.get(page_url, timeout=20)
        resp.raise_for_status()
        page_html = resp.text
    except Exception as e:
        errors.append({"wiki_path": wiki_path, "error": str(e)})
        continue

    infobox_html = find_roundy_infobox(page_html)
    if not infobox_html:
        continue

    file_links = extract_file_links_from_infobox(infobox_html)
    if not file_links:
        continue

    main_thumbnail = file_links[0]
    other_thumbnails = file_links[1:]
    relevant_others = select_relevant_other_thumbnails(main_thumbnail, other_thumbnails)

    print(main_thumbnail, relevant_others)

    results.append(
        {
            "wiki_path": wiki_path,
            "wiki_url": page_url,
            "main_thumbnail": main_thumbnail,
            "main_thumbnail_url": urljoin(BASE_URL, main_thumbnail),
            "relevant_other_thumbnails": relevant_others,
            "relevant_other_thumbnail_urls": [urljoin(BASE_URL, p) for p in relevant_others],
            "all_other_thumbnails": other_thumbnails,
        }
    )

    if idx % 50 == 0:
        print(f"Bearbeitet: {idx}/{len(links_to_process)}")

print(f"Seiten mit Infobox-Thumbnails: {len(results)}")
print(f"Fehler beim Laden: {len(errors)}")

forms_df = pd.DataFrame(results)
forms_df.head(20)

c:\Users\ruedi.luethi\.conda\envs\streamlit\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Gefundene /wiki/-Links: 1195
/wiki/File:PE_Kanto_Map.png []
/wiki/File:JohtoMap.png []
/wiki/File:Hoenn_ORAS.png []
/wiki/File:Sinnoh_BDSP_artwork.png []
/wiki/File:Unova_B2W2_alt.png []
/wiki/File:Kalos_alt.png []
/wiki/File:Alola_USUM_artwork.png []
/wiki/File:Galar_artwork.png []
/wiki/File:Legends_Arceus_Hisui.png []
/wiki/File:Paldea_artwork.png []
/wiki/File:0494Victini.png []
/wiki/File:SilverTitle.png []
/wiki/File:GreenTitle.png []
/wiki/File:0001Bulbasaur.png []
/wiki/File:0002Ivysaur.png []
/wiki/File:0003Venusaur.png ['/wiki/File:0003Venusaur-Mega.png', '/wiki/File:0003Venusaur-Gigantamax.png']
/wiki/File:0004Charmander.png []
/wiki/File:0005Charmeleon.png []
/wiki/File:0006Charizard.png ['/wiki/File:0006Charizard-Mega_X.png', '/wiki/File:0006Charizard-Mega_Y.png', '/wiki/File:0006Charizard-Gigantamax.png']
/wiki/File:0007Squirtle.png []
/wiki/File:0008Wartortle.png []
/wiki/File:0009Blastoise.png ['/wiki/File:0009Blastoise-Mega.png', '/wiki/File:0009Blastoise-Gigantamax.pn

,wiki_path,wiki_url,main_thumbnail,main_thumbnail_url,relevant_other_thumbnails,relevant_other_thumbnail_urls,all_other_thumbnails
0,/wiki/Kanto,https://bulbapedia.bulbagarden.net/wiki/Kanto,/wiki/File:PE_Kanto_Map.png,https://bulbapedia.bulbagarden.net/wiki/File:P...,[],[],[]
1,/wiki/Johto,https://bulbapedia.bulbagarden.net/wiki/Johto,/wiki/File:JohtoMap.png,https://bulbapedia.bulbagarden.net/wiki/File:J...,[],[],[]
2,/wiki/Hoenn,https://bulbapedia.bulbagarden.net/wiki/Hoenn,/wiki/File:Hoenn_ORAS.png,https://bulbapedia.bulbagarden.net/wiki/File:H...,[],[],[]
3,/wiki/Sinnoh,https://bulbapedia.bulbagarden.net/wiki/Sinnoh,/wiki/File:Sinnoh_BDSP_artwork.png,https://bulbapedia.bulbagarden.net/wiki/File:S...,[],[],[]
4,/wiki/Unova,https://bulbapedia.bulbagarden.net/wiki/Unova,/wiki/File:Unova_B2W2_alt.png,https://bulbapedia.bulbagarden.net/wiki/File:U...,[],[],[]
5,/wiki/Kalos,https://bulbapedia.bulbagarden.net/wiki/Kalos,/wiki/File:Kalos_alt.png,https://bulbapedia.bulbagarden.net/wiki/File:K...,[],[],[]
6,/wiki/Alola,https://bulbapedia.bulbagarden.net/wiki/Alola,/wiki/File:Alola_USUM_artwork.png,https://bulbapedia.bulbagarden.net/wiki/File:A...,[],[],[]
7,/wiki/Galar,https://bulbapedia.bulbagarden.net/wiki/Galar,/wiki/File:Galar_artwork.png,https://bulbapedia.bulbagarden.net/wiki/File:G...,[],[],[]
8,/wiki/Hisui,https://bulbapedia.bulbagarden.net/wiki/Hisui,/wiki/File:Legends_Arceus_Hisui.png,https://bulbapedia.bulbagarden.net/wiki/File:L...,[],[],[]
9,/wiki/Paldea,https://bulbapedia.bulbagarden.net/wiki/Paldea,/wiki/File:Paldea_artwork.png,https://bulbapedia.bulbagarden.net/wiki/File:P...,[],[],[]


In [10]:
import json
from pathlib import Path
from urllib.parse import unquote
import re
import pandas as pd

def parse_pokemon_filename(filename: str) -> dict:
    """
    Extrahiert aus Dateinamen oder Wiki-Links wie:
    - 0569Garbodor-Gigantamax.png
    - /wiki/File:0009Blastoise-Mega.png
    generisch: index, name, zusatz.
    """
    text = str(filename).strip()

    # URL-encoding aufloesen (%20 etc.)
    text = unquote(text)

    # Wiki-Praefix entfernen, falls vorhanden
    if text.startswith("/wiki/File:"):
        text = text[len("/wiki/File:"):]
    elif text.startswith("File:"):
        text = text[len("File:"):]

    # Nur Dateiname ohne Pfad, Query, Fragment
    base = Path(text.split("?", 1)[0].split("#", 1)[0]).name
    stem = base.rsplit(".", 1)[0] if "." in base else base

    # Fuehrende Nummer extrahieren
    match = re.match(r"^(?P<index>\d+)(?P<rest>.+)$", stem)
    if not match:
        return {"index": "", "name": stem, "zusatz": ""}

    index = match.group("index")
    rest = match.group("rest")

    # Ersten Suffix-Teil als Zusatz behandeln
    parts = re.split(r"[-_]", rest, maxsplit=1)
    name = parts[0]
    zusatz = parts[1] if len(parts) > 1 else ""

    return {"index": index, "name": name, "zusatz": zusatz}

def _normalize_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip().lower()

def _normalize_nr(value):
    if pd.isna(value):
        return ""
    try:
        return f"{int(value):04d}"
    except Exception:
        s = str(value).strip()
        return s.zfill(4) if s.isdigit() else s

def find_matching_df_edited_indices(df_edited: pd.DataFrame, more: dict) -> list:
    """Match über index (nr) oder name (name_en/name_de)."""
    matched = set()

    target_index = str(more.get("index", "")).strip()
    target_name = _normalize_text(more.get("name", ""))

    if target_index:
        nr_norm = df_edited["nr"].apply(_normalize_nr) if "nr" in df_edited.columns else pd.Series([], dtype=str)
        idx_hits = df_edited.index[nr_norm == target_index].tolist()
        matched.update(idx_hits)

    if target_name:
        for col in ["name_en", "name_de"]:
            if col in df_edited.columns:
                col_norm = df_edited[col].apply(_normalize_text)
                name_hits = df_edited.index[col_norm == target_name].tolist()
                matched.update(name_hits)

    return sorted(matched)

json_path = "../data/pokemon/pokedex_edited.json"
df_edited = pd.read_json(json_path, orient="records")

# if "weitere_erscheinungsformen" not in df_edited.columns:
#     df_edited["weitere_erscheinungsformen"] = None

updates = 0
no_match = []

for _, row in forms_df.iterrows():
    thumbs = row.get("relevant_other_thumbnails", [])
    if not isinstance(thumbs, list):
        continue

    for thumb in thumbs:
        more = parse_pokemon_filename(thumb)

        # Nur Einträge mit erkanntem Index übernehmen
        if not str(more.get("index", "")).strip():
            continue

        target_indices = find_matching_df_edited_indices(df_edited, more)
        if not target_indices:
            no_match.append({"thumb": thumb, "parsed": more})
            continue

        payload = {
            "form_name": more.get("zusatz", ""),
            "link_bulbpedia": thumb,
        }

        for i in target_indices:
            existing = df_edited.at[i, "weitere_erscheinungsformen"]

            if not isinstance(existing, list):
                existing = []

            # Duplikat über link_bulbpedia vermeiden
            exists = any(
                isinstance(x, dict) and x.get("link_bulbpedia") == payload["link_bulbpedia"]
                for x in existing
            )
            if not exists:
                existing.append(payload)
                df_edited.at[i, "weitere_erscheinungsformen"] = existing
                updates += 1

print(f"Hinzugefügte Formen-Einträge: {updates}")
print(f"Ohne Match in df_edited: {len(no_match)}")

# Vorschau auf Zeilen mit mindestens einer weiteren Erscheinungsform
# preview = df_edited[df_edited["weitere_erscheinungsformen"].apply(lambda x: isinstance(x, list) and len(x) > 0)]
# preview[["nr", "name_en", "name_de", "weitere_erscheinungsformen"]].head(20)
# preview.head(20)
# print(json.dumps(preview[["nr", "name_en", "name_de", "weitere_erscheinungsformen"]].head(20).to_dict(orient="records"), indent=2, ensure_ascii=False))

df_edited.to_json("../data/pokemon/pokedex_extended.json", orient="records", force_ascii=False, indent=2)

Hinzugefügte Formen-Einträge: 274
Ohne Match in df_edited: 0


In [39]:
# Alle link_bulbpedia aus df_edited -> weitere_erscheinungsformen ausgeben
for _, row in df_edited[0:10].iterrows():
    forms = row.get("weitere_erscheinungsformen")

    if not isinstance(forms, list):
        continue

    
    print(forms)


    for form in forms:
        if not isinstance(form, dict):
            continue

        link = form.get("link_bulbpedia")
        if isinstance(link, str) and link.strip():
            print(link)

[{'norm_name': 'Mega', 'link_bulbpedia': '/wiki/File:0003Venusaur-Mega.png'}, {'norm_name': 'Gigantamax', 'link_bulbpedia': '/wiki/File:0003Venusaur-Gigantamax.png'}]
/wiki/File:0003Venusaur-Mega.png
/wiki/File:0003Venusaur-Gigantamax.png
[{'norm_name': 'Mega_X', 'link_bulbpedia': '/wiki/File:0006Charizard-Mega_X.png'}, {'norm_name': 'Mega_Y', 'link_bulbpedia': '/wiki/File:0006Charizard-Mega_Y.png'}, {'norm_name': 'Gigantamax', 'link_bulbpedia': '/wiki/File:0006Charizard-Gigantamax.png'}]
/wiki/File:0006Charizard-Mega_X.png
/wiki/File:0006Charizard-Mega_Y.png
/wiki/File:0006Charizard-Gigantamax.png


In [12]:
import re
import requests
from urllib.parse import urljoin
from pathlib import Path

BASE_BULBA = "https://bulbapedia.bulbagarden.net"


def extract_original_download_link(file_page_html: str, page_url: str) -> str:
    # 1) Häufigster Fall auf MediaWiki-File-Seiten
    m = re.search(
        r'<div[^>]*class=["\\"][^"\\"]*fullImageLink[^"\\"]*["\\"][^>]*>\s*<a[^>]*href=["\\"]([^"\\"]+)["\\"]',
        file_page_html,
        flags=re.IGNORECASE,
    )
    if m:
        return urljoin(page_url, m.group(1))

    # 2) Fallback: Linktext 'Original file' / 'Original-Datei'
    m = re.search(
        r'<a[^>]*href=["\\"]([^"\\"]+)["\\"][^>]*>\s*(?:Original file|Original-Datei)\s*</a>',
        file_page_html,
        flags=re.IGNORECASE,
    )
    if m:
        return urljoin(page_url, m.group(1))

    # 3) Fallback: erster direkter Upload-Link
    m = re.search(r'href=["\\"](https?://[^"\\"]*/upload/[^"\\"]+)["\\"]', file_page_html, flags=re.IGNORECASE)
    if m:
        return m.group(1)

    return ""


sess = requests.Session()
sess.headers.update({"User-Agent": "Mozilla/5.0 (compatible; bulba-original-link-extractor/1.0)"})

updated = 0
missing = 0
errors = 0
seen = set()

for row_idx, row in df_edited.iterrows():
    forms = row.get("weitere_erscheinungsformen")
    if not isinstance(forms, list):
        continue

    for form in forms:
        if not isinstance(form, dict):
            continue

        link = form.get("link_bulbpedia")
        if not isinstance(link, str) or not link.strip():
            continue

        link = link.strip()
        file_page_url = urljoin(BASE_BULBA, link)

        # Optionales Request-Deduping pro Datei-Link
        if link in seen and form.get("sprite_bulbapedia"):
            continue

        try:
            r = sess.get(file_page_url, timeout=20)
            r.raise_for_status()
            original_url = extract_original_download_link(r.text, file_page_url)
        except Exception as e:
            errors += 1
            print(f"Fehler bei {file_page_url}: {e}")
            continue

        seen.add(link)

        if original_url:
            # Direkt im passenden Eintrag setzen
            form["sprite_bulbapedia"] = original_url
            updated += 1
            print(f"Gesetzt: {row.get('name_de', row.get('name_en', row_idx))} -> {original_url}")
        else:
            missing += 1
            print(f"Kein Original-Link gefunden: {file_page_url}")

print(f"Aktualisiert: {updated}, Kein Link gefunden: {missing}, Fehler: {errors}")

# Ergebnis persistieren
json_out = Path("../data/pokemon/pokedex_extended_right_link.json")
df_edited.to_json(json_out, orient="records", force_ascii=False, indent=2)
print(f"Gespeichert: {json_out.resolve()}")

Gesetzt: Bisaflor -> https://archives.bulbagarden.net/media/upload/f/f7/0003Venusaur-Mega.png
Gesetzt: Bisaflor -> https://archives.bulbagarden.net/media/upload/4/45/0003Venusaur-Gigantamax.png
Gesetzt: Glurak -> https://archives.bulbagarden.net/media/upload/3/35/0006Charizard-Mega_X.png
Gesetzt: Glurak -> https://archives.bulbagarden.net/media/upload/a/a0/0006Charizard-Mega_Y.png
Gesetzt: Glurak -> https://archives.bulbagarden.net/media/upload/0/0c/0006Charizard-Gigantamax.png
Gesetzt: Turtok -> https://archives.bulbagarden.net/media/upload/6/6e/0009Blastoise-Mega.png
Gesetzt: Turtok -> https://archives.bulbagarden.net/media/upload/b/bd/0009Blastoise-Gigantamax.png
Gesetzt: Smettbo -> https://archives.bulbagarden.net/media/upload/e/ee/0012Butterfree-Gigantamax.png
Gesetzt: Bibor -> https://archives.bulbagarden.net/media/upload/7/73/0015Beedrill-Mega.png
Gesetzt: Tauboss -> https://archives.bulbagarden.net/media/upload/f/f6/0018Pidgeot-Mega.png
Gesetzt: Rattfratz -> https://archives.bu

In [18]:
json_path = "../data/pokemon/pokedex_extended_right_link.json"
df_edited = pd.read_json(json_path, orient="records")

removed_entries = 0
merged_types = 0
rows_changed = 0

for row_idx, row in df_edited.iterrows():
    more_forms = row.get("weitere_erscheinungsformen")
    more_devs = row.get("weitere_entwicklungen")

    forms_list = more_forms if isinstance(more_forms, list) else []
    devs_list = more_devs if isinstance(more_devs, list) else []

    # Gemeinsame Sicht auf beide Listen
    all_forms = []
    for i, entry in enumerate(forms_list):
        if isinstance(entry, dict):
            all_forms.append(("forms", i, entry))
    for i, entry in enumerate(devs_list):
        if isinstance(entry, dict):
            all_forms.append(("devs", i, entry))

    # Nach sprite_bulbapedia gruppieren
    by_sprite = {}
    for source, idx, entry in all_forms:
        sprite = entry.get("sprite_bulbapedia")
        if not isinstance(sprite, str) or not sprite.strip():
            continue
        key = sprite.strip()
        by_sprite.setdefault(key, []).append((source, idx, entry))

    to_remove_forms = set()
    to_remove_devs = set()
    changed_this_row = False

    for _, group in by_sprite.items():
        if len(group) < 2:
            continue

        with_link = [g for g in group if isinstance(g[2].get("link_bulbpedia"), str) and g[2].get("link_bulbpedia").strip()]
        without_link = [g for g in group if not (isinstance(g[2].get("link_bulbpedia"), str) and g[2].get("link_bulbpedia").strip())]

        if not with_link or not without_link:
            continue

        # Behalte den ersten Eintrag mit link_bulbpedia
        keep_source, keep_idx, keep_entry = with_link[0]

        for drop_source, drop_idx, drop_entry in without_link:
            # typ1/typ2 vom zu löschenden Eintrag übernehmen, falls im Keep-Eintrag leer
            for typ_key in ["typ1", "typ2"]:
                drop_val = drop_entry.get(typ_key)
                keep_val = keep_entry.get(typ_key)

                drop_has = isinstance(drop_val, str) and drop_val.strip()
                keep_has = isinstance(keep_val, str) and keep_val.strip()

                if drop_has and not keep_has:
                    keep_entry[typ_key] = drop_val
                    merged_types += 1

            if drop_source == "forms":
                to_remove_forms.add(drop_idx)
            else:
                to_remove_devs.add(drop_idx)

            removed_entries += 1
            changed_this_row = True

    if to_remove_forms:
        forms_list = [entry for i, entry in enumerate(forms_list) if i not in to_remove_forms]
        df_edited.at[row_idx, "weitere_erscheinungsformen"] = forms_list

    if to_remove_devs:
        devs_list = [entry for i, entry in enumerate(devs_list) if i not in to_remove_devs]
        df_edited.at[row_idx, "weitere_entwicklungen"] = devs_list

    if changed_this_row:
        rows_changed += 1

print(f"Geänderte Zeilen: {rows_changed}")
print(f"Entfernte Duplikat-Einträge ohne link_bulbpedia: {removed_entries}")
print(f"Übernommene typ1/typ2-Werte: {merged_types}")

out_path = "../data/pokemon/pokedex_extended_cleaned.json"
df_edited.to_json(out_path, orient="records", force_ascii=False, indent=2)
print(f"Gespeichert: {out_path}")

Geänderte Zeilen: 67
Entfernte Duplikat-Einträge ohne link_bulbpedia: 79
Übernommene typ1/typ2-Werte: 77
Gespeichert: ../data/pokemon/pokedex_extended_cleaned.json
